In [5]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
# from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

In [3]:
LITE_MODE = True

In [6]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [7]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [13]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    
    for row in reader:
        human_predictions.append(float(row[1]))

In [14]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [15]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")

Human predicted 120.0 for an item that actually costs 219.0


In [ ]:
evaluate(human_pricer, test, size=100)

### Neural Network

In [23]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]
print(y[0])

64.3


In [25]:
# Set the random seed for reproducibility
np.random.seed(42)

# Create the HashingVectorizer with specific parameters
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)

# Fit the vectorizer to the documents and transform them into feature matrix X
X = vectorizer.fit_transform(documents)

In [26]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [27]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [28]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [ ]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

In [30]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
evaluate(neural_network, test)

In [32]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [33]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [34]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [39]:
from openai import OpenAI

base_url = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

openai = OpenAI(base_url=base_url, api_key=openrouter_api_key)
def gpt_4__1_nano(item):
    response = openai.chat.completions.create(
        model="openai/gpt-4.1-nano",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content


In [47]:
gpt_4__1_nano(test[0])

'$250'

In [48]:
test[0].price

219.0

In [ ]:
evaluate(gpt_4__1_nano, test)

In [49]:
def claude_opus_4_5(item):
    response = openai.chat.completions.create(
        model="anthropic/claude-opus-4-5",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content


print(claude_opus_4_5(test[0]))

$199


In [ ]:
evaluate(claude_opus_4_5, test)

In [56]:
def gemini_3_pro_preview(item):
    response = openai.chat.completions.create(
        model="google/gemini-2.5-pro-preview-03-25",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content

In [57]:
print(gemini_3_pro_preview(test[0]))

$209


In [66]:
def grok_2_1(item):
    response = openai.chat.completions.create(
        model="x-ai/grok-3-beta",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content

In [67]:
print(grok_2_1(test[0]))


$229


In [68]:
test[0].price

219.0

In [76]:
def gpt_5_1(item):
    response = openai.chat.completions.create(
        model="openai/gpt-4.1",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content

In [77]:
print(gpt_5_1(test[0]))

$199
